# Knihovna funkcí a modulů pro neuronové sítě

- V tomto cvičení si navrhneme vlastní zjednodušenou knihovnu `ans.nn` pro neuronové sítě, která bude svojí architekturou napodobovat PyTorch.
- Bude založená na objektovém návrhu s třemi základními třídami
  1. `Module` bude základní třídou pro "vrstvy" v síti.
  2. `Optimizer` bude základní třídou pro (gradientní) optimalizační metody (v Pytorch v submodulu `torch.optim`).
  3. `Function` bude základní třídou pro složitější diferencovatelné operace, které nejsou vhodné pro implementaci přímo ve `Variable` (v PyTorch v submodulu `torch.nn.functional`).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname('neural_library.ipynb')))

In [3]:
from typing import Any, Callable
import sys
sys.path.append('..')  # import tests

import numpy as np
import matplotlib.pyplot as plt
import PIL
import torch
import torchvision
import ans
import ans.nn
from tests import test_neural_library

# Vrstvy v síti jako třída `Module`

- V minulých cvičeních jsme klasifikační modely definovali jako třídy v modulu `ans.classification` s *napevno* danými parametry a modelem klasifikace.
- Tento způsob samozřejmě není příliš flexibilní a pokud bychom chtěli např. třívrstvý místo dvouvrstvého perceptronu, museli bychom definovat novou třídu s odpovídajícími parametry.
- V PyTorch je flexibilita zajištěna objektovým návrhem: každá vrstva je objekt s definovaným dopředným průchodem a parametry.
- Takto definované vrstvy potom lze jako základní bloky libovolně skládat za sebe. Viz [příklad z pytorch.org](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
  ``` python
  model = nn.Sequential(
      nn.Conv2d(1,20,5),
      nn.ReLU(),
      nn.Conv2d(20,64,5),
      nn.ReLU()
  )
  ```
  kde výsledný model sestává z konvoluce následované ReLU následované opět konvolucí a ReLU.

**Třída `Module`**
- Po vzoru PyTorch všechny vrstvy budou odvozeny od `Module`.
- Každá vrstva odvozená od `Module`
  - zajistí vytvoření a inicializaci svých parametrů v `__init__`,
    - protože budeme potřebovat gradienty, každý učitelný parametr vrstvy bude typu `Variable` (PyTorch má speciální třídu `Parameter`, což je `Tensor` obalený vlastnostmi navíc).
  - bude mít definovaný dopředný průchod v metodě `forward`
    - vstupem je jedna nebo více `Variable` a výstupem jedna `Variable` (v PyTorch neomezeno).
  - bude mít atribut `training: bool`
    - který říká, zda se modul nachází v trénovacím či evaluačním módu
    - a který je nutný pro vrstvy jako batch normalizace či dropout.
  - bude umět vrátit seznam všech svých parametrů zavoláním `named_parameters()`
    - což je metoda, která vrátí `list` dvojic (jméno, parametr) pro každý parametr vrstvy,
    - parametrem se zde myslí např. váhová matice nebo bias vektor,
- Každá vrstva typu `Module` bude moci jako atributy obsahovat jiné vrstvy odvozené od `Module`.
  - Takto bude možné ze základních bloků sestavovat libovolně složitou "vrstvu" až po celé neurosítě, které budou rovněž samy `Module`.
  - Např. seznam parametrů potom musí vrátit parametry nejen "sebe", ale i vnořených vrstev. Analogicky platí pro ostatní metody, jenž musí rovněž rekurzivně projít i vnořené moduly.
- Třída obsahuje ještě další metody, např.
  - `to(dtype: Optional[torch.dtype] = None, device: Optional[str] = None)`
    - která přenese celý modul (jeho parametry) např. na GPU (`device='cuda'`) a to včetně všech vnořených submodulů,
  - `train` a `eval`
    - pro přepínání atributu `training` i pro vnořené moduly,
  - a další, viz [`ans.nn`](../ans/nn.py).

## Vrstva `Linear`

- Třída obsahuje parametry klasifikátoru jako atributy  
  | atribut  | značení                                                 | rozměr       |
  |----------|---------------------------------------------------------|--------------|
  | `weight` | $\boldsymbol{w} = \left[w_{d,k}\right]$                | $D \times K$ |
  | `bias`   | $\boldsymbol{b} = \left[b_1^{(1)}, \ldots, b_H\right]$ | $K$          |
- Parametry inicializujte v `Linear.__init__`.
- Váhovou matici *i bias* inicializujte metodou Xavier/Glorot/He s uniformním rozdělením, tj.
  $$
  w_{d,c} \sim \mathcal{U}\left(\frac{-1}{\sqrt D}, \frac{+1}{\sqrt D}\right)
  $$
- Dopředný průchod je
  $$
  \boldsymbol{z} = \boldsymbol{x} \cdot \boldsymbol{w} + \boldsymbol{b}
  $$

**Poznámka**
- Bias budeme inicializovat na náhodné hodnoty proto, aby bylo možné modul přímo porovnat s `torch.nn.Linear`.
- V metodě `Linear.forward` pouze využijte operátory `@` a `+`, které jste implementovali v minulých cvičeních.

### TODO: implementuje třídu [`ans.nn.Linear`](../ans/nn.py).

In [4]:
test_neural_library.TestLinearModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.092s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Vrstva `Sigmoid`

- Sigmoid nemá žádné parametry a není potřeba inicializovat.
- Dopředný průchod je
  $$
  z = \frac{1}{1+e^{-x}}
  $$
- Opět stačí využít již hotovou metodu `Variable.sigmoid`.

### TODO: implementuje třídu [`ans.nn.Sigmoid`](../ans/nn.py).

In [5]:
test_neural_library.TestSigmoidModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.239s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Vrstva `ReLU`

- Dopředný průchod je
  $$
  z = \max(0, x)
  $$
- Jelikož jsme ReLU odposud nepoužívali, je nutné operaci nejprve implementovat (forward a backward průchody).
- Pro svou jednoduchost se ReLU hodí vedle `Variable.sigmoid` metody v `ans.autograd`. Budeme jí proto používat např. jako `x.relu()`.
- Až bude hotová implementace `Variable.relu()`, implementujte ReLU i jako vrstvu `ans.nn.ReLU`.

**Poznámka**
- Při jakékoliv změně `ans/autograd.py` bude opět potřeba restartovat kernel notebooku kvůli tomu, jak funguje autoreload extension.
- S ostatními soubory restart při změně nutný není a vše se automagicky reloaduje správně a samo.

### TODO: implementuje metodu [`ans.autograd.Variable.relu`](../ans/autograd.py).

In [6]:
test_neural_library.TestReLUVariable.eval()

test_operation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_operation) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.014s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### TODO: implementuje třídu [`ans.nn.ReLU`](../ans/nn.py).

In [7]:
test_neural_library.TestReLUModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Sekvenční aplikace vrstev: vrstva `Sequential`

- Vytvoříme třídu `ans.nn.Sequential`, která
  - v `__init__` převezme seznam vrstev (`list[Module]`)
  - a ve `forward(x: Variable)` je stejném pořadí jednu po druhé aplikuje na vstup `x` a vrátí výsledek.
- Třída bude imitovat [`torch.nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html), viz příklad výše.

### TODO: implementuje třídu [`ans.nn.Sequential`](../ans/nn.py).

In [8]:
test_neural_library.TestSequentialModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.011s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# Optimalizace jako třída `Optimizer`

- Optimalizace bude rovněž vyjádřena jako třída.
- Všechny metody optimalizace (SGD, Adam, ...) budou odvozeny od základní třídy [`ans.nn.Optimizer`](../ans/nn.py).
- Objekt typu `Optimizer`
  - při vytváření do `__init__` převezme seznam parametrů (viz metoda `Module.named_parameters()`), které se budou optimalizovat.
    - Viz příklad na [pytorch.org](https://pytorch.org/docs/stable/optim.html)
      ``` python
      optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
      ```
  - a implementuje metodu `step`
    - která po zavolání provede vždy jeden update parametrů (obvykle se funkce volá po výpočtu gradientů v každé minidávce),
    - a kde se liší různé metody optimalizace mezi sebou.

## Metoda SGD jako `Optimizer`

- Po vzoru knihovny PyTorch budeme implementovat SGD verzi s "hybností", tzv. Momentum SGD, včetně regularizace.

**Jeden krok optimalizace**

$$
\begin{split}
    q_{t} & := \overline{\theta}_t + \lambda \cdot \theta_{t} \\
    v_{t} & := \alpha \cdot v_{t-1} - \gamma \cdot q_{t} \\
    \theta_{t+1} & := \theta_{t} + v_t
\end{split}
$$
- kde
  - $\theta_{t}$ a $\theta_{t+1}$ značí původní, resp. nově vypočtenou hodnotu jednoho z parametrů modelu (např. jednoho z prvků váhové matice)
  - $\overline{\theta}_t = \partial l_t / \partial \theta_{t}$ je gradient celkového lossu $l_t$ v iteraci $t$ vůči parametru $\theta_{t}$ získaný zpětnou propagací
  - $v_{t-1}$ a $v_t$ značí "rychlosti" (velocity) z minulého, resp. aktuálního kroku
  - $\alpha \in [0, 1]$ (hyperparametr) je reálné číslo mezi 0 a 1 (včetně) a značí tzv. hybnost (momentum)
  - $\lambda$ (hyperparametr) je koeficient L2 regularizace, tzv. weight decay
  - $\gamma$ (hyperparametr) značí krok učení, tzv. learning rate
- Uvedené update pravidlo se provede pro všechny parametry předané seznamu předaného do `__init__`.

### TODO: implementuje třídu [`ans.nn.SGD`](../ans/nn.py).

In [9]:
test_neural_library.TestSGD.eval()

test_init (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_init) ... ok
test_momentum (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_momentum) ... ok
test_sgd (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_sgd) ... ok
test_weight_decay (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_weight_decay) ... ok
test_weight_decay_momentum (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_weight_decay_momentum) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.030s

OK


<unittest.runner.TextTestResult run=5 errors=0 failures=0>

## (Bonus) Metoda Adam jako `Optimizer`

**Jeden krok optimalizace**

$$
\begin{split}
    q_t & := \overline{\theta}_t + \lambda \cdot \theta_t \\
    m_t & := \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot q_t \\
    v_t & := \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot q_t^2 \\
    \mu_t & := \frac{m_t}{1 - \beta_1^t} \\
    \nu_t & := \frac{v_t}{1 - \beta_2^t} \\
    \theta_{t+1} & := \theta_t - \gamma \cdot \frac{\mu_t}{\sqrt{\nu_t} + \epsilon}
\end{split}
$$
- kde
  - všechny proměnné jsou reálná čísla a mají podobný význam jako u SGD
  - $v$ a $u$ jsou buffery, které se mezi jednotlivými iteracemi předávají
- Hyperparametry metody jsou
  - learning rate $\gamma$
  - weight decay $\lambda$
  - $\beta_1 \in [0, 1]$
  - $\beta_2 \in [0, 1]$
  - epsilon $\epsilon \approx 10^{-8}$

### TODO: implementuje třídu [`ans.nn.Adam`](../ans/nn.py).

In [10]:
test_neural_library.TestAdam.eval()

test_adam (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_adam) ... ok
test_init (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_init) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.019s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Diferencovatelné operace jako třída `Function`

- Přepisování operátorů u `Variable` je vhodné pro základní a jednoduché operace jako sčítání, násobení nebo sigmoid.
- Pro složitější operace je přehlednější refaktorovat dopředný a zpětný průchod do dvou oddělených `*_forward` a `*_backward` funkcí.
- Navíc nám to umožní zautomatizovat "biolerplate", kdy musíme v každé metodě ručně vytvářet instance `Variable` se správnou `grad_fn`, `parents` atd.
- Všechny diferencovatelné operace bude reprezentovat základní třída [`Function`](../ans/nn.py), jež má tři metody
  1. `forward(*inputs: Tensor, ...)`
     - která definuje dopředný průchod
     - a pracuje pouze s `Tensor`y, tj. nikoliv `Variable`,
     - a vrátí dvojici `(výsledek: Tensor, cache: Any)` (`cache` slouží jako mezipaměť pro zpětný průchod)
  2. `backward(dout, cache)`
     - která definuje zpětný průchod (řetízkové pravidlo),
     - opět pracuje pouze s `Tensor` (neřeší `Variable`)
  3. `apply(*inputs: Union[torch.Tensor, Variable])`
     - která obaluje dopředný průchod `forward` o boilerplate,
     - vstupy má buď `Tensor` nebo `Variable`,
     - zavolá `forward`, převezme výsledek (a cache) a vytvoří výstupní `Variable` se všemi potřebnými náležitostmi.
     - výsledná `Variable.grad_fn` bude počítat gradienty pouze na ty vstupy, které byly do `apply` předány jako `Variable`.
- *Jako uživatel budeme funkce odvozené od `Function` volat přes `.apply()`.*

## (Bonus) Dropout

- Budeme implementovat tzv. [inverted dropout](https://stats.stackexchange.com/questions/205932/dropout-scaling-the-activation-versus-inverting-the-dropout), tedy verzi, kdy ke škálování výstupu dochází již v trénovací fázi a eval režimu se chová jako identita.

**Dopředný průchod v *trénovacím* režimu**

$$
\begin{split}
    m & \sim \mathcal{U}\left[ 0, 1 \right] \\
    z & = \begin{cases}
        0 & \textrm{pokud} & m \lt p \\
        \frac{x}{1 - p} & \textrm{pokud} & m \ge p \\
    \end{cases}
\end{split}
$$
- $x$ je reálné číslo (skalár)
- $z$ je reálné číslo (skalár)
- $m$ je reálné číslo náhodně vybrané z intervalu $[0, 1]$
- $p$ je reálné číslo (skalár) určující pravděpodobnost, s jakou dojde k vynulování $x$

**Zpětný průchod v *trénovacím režimu***

$$
\overline{x} = \begin{cases}
    0 & \textrm{pokud} & m \lt p \\
    \frac{\overline{z}}{1 - p} & \textrm{pokud} & m \ge p \\
\end{cases}
$$
- $\overline{z}$ je příchozí gradient na $z$

### TODO: implementuje třídu [`ans.nn.DropoutFunction`](../ans/nn.py).

In [11]:
test_neural_library.TestDropoutFunction.eval()

test_function (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_function) ... ok
test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.033s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje třídu [`ans.nn.Dropout`](../ans/nn.py).

In [12]:
test_neural_library.TestDropoutModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.171s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## (Bonus) BatchNorm1d

- Budeme implementovat 1D verzi, která pracuje pouze s vektory.
- Úkolem je implementovat normalizaci dávky nejprve jako funkce typu `Function` a poté jako `Module` (aby šla použít např. jako vrstva do `Sequential`).

**Parametry**

| atribut        | typ                     | značení               | rozměr | poznámka                                         |
|----------------|-------------------------|-----------------------|--------|--------------------------------------------------|
| `weight`       | `ans.autograd.Variable` | $\boldsymbol{\gamma}$ | $D$    | nepovinné, použije se, pouze pokud `affine=True` |
| `bias`         | `ans.autograd.Variable` | $\boldsymbol{\beta}$  | $D$    | nepovinné, použije se, pouze pokud `affine=True` |
| `running_mean` | `torch.Tensor`          | $\boldsymbol{m}$      | $D$    | není trénovatelný parametr (nemá gradient)       |
| `running_var`  | `torch.Tensor`          | $\boldsymbol{v}$      | $D$    | není trénovatelný parametr (nemá gradient)       |

**Dopředný průchod v ***trénovacím*** režimu**

$$
\begin{split}
    \boldsymbol{\mu} & = \frac{1}{N}\sum_{n=1}^N{\boldsymbol{x}_n} \\
    \boldsymbol{\sigma}^2 & = \frac{1}{N}\sum_{n=1}^N{\left(\boldsymbol{x}_n - \boldsymbol{\mu}\right)^2} \\
    \widehat{\boldsymbol{x}}_n & = \frac{\boldsymbol{x}_n - \boldsymbol{\mu}}{\sqrt{\boldsymbol{\sigma}^2 + \epsilon}} \\
    \boldsymbol{z}_n & = \boldsymbol{\gamma} \odot \widehat{\boldsymbol{x}}_n + \boldsymbol{\beta}
\end{split}
$$
kde
- $\boldsymbol{x}_n = [x_{n,1}, \ldots, x_{n,D}]$ je jeden vzorek dávky jako (řádkový) vektor s rozměrem $D$
- $N$ je počet vstupů (vektorů) v dávce
- $\boldsymbol{\sigma}^2$ je *vychýlený* odhad vektoru rozptylů na základě dávky
- $\epsilon \approx 10^{-5}$ je konstanta (reálné číslo) zabraňující dělení nulou
- $\boldsymbol{z}_n = [z_{n,1}, \ldots, z_{n,D}]$ je (řádkový) výstupní vektor s rozměrem $D$
- $\odot$ značí prvkové pronásobení

V trénovacím režimu zároveň průběžně akumulujeme statistiky průměrného vektoru a vektoru rozptylů

$$
\begin{split}
    \boldsymbol{m} & := (1 - \alpha) \cdot \boldsymbol{m} + \alpha \cdot \boldsymbol{\mu} \\
    \boldsymbol{v}^2 & := (1 - \alpha) \cdot \boldsymbol{v}^2 + \alpha \cdot \frac{N}{N-1} \cdot \boldsymbol{\sigma^2}
\end{split}
$$
- $\boldsymbol{m}$ je průběžný odhad očekávaného vektoru $\boldsymbol{\mu}$, tj. $\boldsymbol{m} \approx E[\boldsymbol{\mu}]$
- $\boldsymbol{v}^2$ je průběžný *nevychýlený* odhad očekávaného vektoru $\boldsymbol{\sigma^2}$, tj. $\boldsymbol{v}^2 \approx E[\boldsymbol{\sigma^2}]$
- $\alpha$ je vyhlazovací koeficient ("momentum") pamatování při odhadu průběžného průměru a rozptylu
- $N / (N - 1)$ je tzv. Besselova korekce, viz pozn. dole

**Zpětný průchod**
$$
\begin{split}
    \overline{\boldsymbol{x}}_n & = 
        \frac{\boldsymbol{\gamma}}{\sqrt{\boldsymbol{\sigma}^2 + \epsilon}} \odot \left(
            \overline{\boldsymbol{z}}_n
            - \frac{1}{N}\sum_{i=1}^N{\overline{\boldsymbol{z}}_i}
            - \frac{1}{N} \widehat{\boldsymbol{x}}_n \odot \sum_{i=1}^N{\overline{\boldsymbol{z}}_i\odot\widehat{\boldsymbol{x}}_i}
        \right) \\
    \overline{\boldsymbol{\gamma}} & = \sum_{n=1}^N{ \overline{\boldsymbol{z}}_n \odot \widehat{\boldsymbol{x}}_n } \\
    \overline{\boldsymbol{\beta}} & =\sum_{n=1}^N{ \overline{\boldsymbol{z}}_n }
\end{split}
$$

**Dopředný průchod v ***eval*** režimu**

- V eval režimu nepočítáme statistiky z dávky, ale použijeme průběžné odhady $\boldsymbol{m}$ a $\boldsymbol{v}^2$ z trénovací fáze.
- Zároveň nedochází k jejich updatu.
  $$
  \begin{split}
      \widehat{\boldsymbol{x}}_n & = \frac{\boldsymbol{x}_n - \boldsymbol{m}}{\sqrt{\boldsymbol{v}^2 + \epsilon}} \\
      \boldsymbol{z}_n & = \boldsymbol{\gamma} \odot \widehat{\boldsymbol{x}}_n + \boldsymbol{\beta}
  \end{split}
  $$

**Zpětný průchod v ***eval*** režimu**
$$
\overline{\boldsymbol{x}}_n = \frac{\boldsymbol{\gamma}}{\sqrt{\boldsymbol{v}^2 + \epsilon}} \odot \overline{\boldsymbol{z}}_n
$$

**Vychýlený a nevychýlený odhad rozptylu**
- V PyTorchi pozor na odhad rozptylu metodou `var`.
- Ve výchozím režimu počítá tzv. *nevychýlený* odhad, kdy se namísto $1/N$ dělí $1/(N-1)$, viz [Besselova korekce](https://en.wikipedia.org/wiki/Bessel%27s_correction).
- Batch normalizace ale přitom používá *vychýlený* odhad $\boldsymbol{\sigma}^2$ a do funkce [torch.var](https://pytorch.org/docs/stable/generated/torch.var.html) je proto nutné explicitně zadat `unbiased=False`.
- Při výpočtu průběžného rozptylu $\boldsymbol{v^2}$ Pytorch přenásobí odhad $\boldsymbol{\sigma}^2$ Besselovou korekcí $N / (N - 1)$. Nejedná se o nekonzistentní chování ani chybu. Důvod je ten,  že $\boldsymbol{v^2}$ se snaží odhadovat skutečný očekávaný rozptyl dávky, tzn. $\boldsymbol{v}^2 \approx E[\boldsymbol{\sigma^2}]$, a ten je nejpřesnější jako nevychýlený.

### TODO: implementuje třídu [`ans.nn.BatchNorm1dFunction`](../ans/nn.py).

In [13]:
test_neural_library.TestBatchNorm1dFunction.eval()

test_function (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_function) ... ok
test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.014s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje třídu [`ans.nn.BatchNorm1d`](../ans/nn.py).

In [14]:
test_neural_library.TestBatchNorm1dModule.eval()

test_module (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_module) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.023s

OK


APAR: bias, grad: tensor([-11.6950,  19.0859,   1.6395,   0.4678,   1.1334, -16.6560,   6.7304,
        -10.7450,  -9.7216,   7.3778,  -0.1985, -13.7474, -11.6661, -12.8044,
         -4.8328, -11.1881,  -6.3662,  -3.5894, -12.1990,  -4.7852,  -0.3494,
         -0.1202,   4.5206,  -1.9180,  14.7856,   6.8104,  -7.6775,  -8.3959,
          7.3886,  -0.5391,  -7.5588,  -5.4372,   8.3790,  -9.2427,   1.8773,
          7.8766,   4.0140, -11.7359,  17.1909,   1.1627]), grad_fn: None
APAR: weight, grad: tensor([ -9.9447,  -4.9425,   6.6643,  -3.1687,  -9.1784,   3.3400,  12.0967,
          5.3772, -10.9297,   4.3976,  -4.5074,   1.8506,   0.4765,  25.3610,
         -3.1238,   2.3752,   7.7374,   5.7735,   7.0107,  11.5109,   0.6520,
         -0.3148, -13.2050,  -5.8882,  -6.6204,  11.5805,   8.7998,   0.7674,
          8.8609,  -9.9678,  12.7346,   8.2757,   4.1425,   4.2316, -19.3030,
         -0.7124, -17.7268,  -3.5422,  -1.0232,   5.6181],
       grad_fn=<SumBackward1>), grad_fn: None
APA

<unittest.runner.TextTestResult run=1 errors=0 failures=0>

    # Obecný klasifikační model: `AutogradClassifier`

- Vytvoříme třídu `ans.classification.AutogradClassifier(ans.nn.Module)`, která bude mít dva parametry
  1. `backbone: Module`, což bude vrstva zajišťující výpočet skóre (logitů) ze vstupu,
  2. `optimizer: Optimizer`, který bude provádět update parametrů (učení sítě).
- Parametr `backbone` bude např. `Sequential` aplikující např. `Linear` --> `Sigmoid` --> `Linear`.
- Poté bude třída implementovat opět `train_step` a `val_step` se stejným významem a API jako v minulých cvičeních.
- Třída je sama odvozena od `ans.nn.Module`, tj. lze na ní volat `train()`, `eval()`, `named_parameters()` a další.

**Poznámka**
- Nezapomeňte před `loss.backprop()` vynulovat gradienty!

### TODO: implementujte třídu [`ans.classification.AutogradClassifier`](../ans/classification.py).

In [15]:
test_neural_library.TestAutogradClassifier.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... FAIL
test_train_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_train_step) ... ERROR
test_val_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_step) ... ERROR

ERROR: test_train_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_train_step)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 537, in test_train_step
    loss, logits = self.model.train_step(self.inputs, self.targets)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../ans/classification.py", line 423, in train_step
    raise NotImplementedError
NotImplementedError

ERROR: test_val_step (tests.ANSTestCase.eval.<locals>._Tes

<unittest.runner.TextTestResult run=3 errors=2 failures=1>

## Úprava `train_epoch` a `validate`

- Chování vrstev se může lišit v závislosti na režimu (train vs eval), viz např. `Dropout` nebo `BatchNorm1d`.
- Existující funkce `train_epoch` a `validate` v submodulu `ans.classification` s tímto nepočítají.
- Doplňte proto na začátek obou funkcí automatické přepnutí modelu do správného módu tak, abychom to nemuseli provádět vždy před jejich závoláním manuálně.
- Nerozbijte přitom zpětnou kompatibilitu s modely `LinearSoftmaxModel` a ost. z minulých cvičení, které metody `train()` a `eval()` nemají.

### TODO: upravte metody [`ans.classification.train_epoch`](../ans/classification.py) a [`ans.classification.validate`](../ans/classification.py).

In [16]:
test_neural_library.TestTrainEpochValidate.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... FAIL
test_train_epoch (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_train_epoch) ... ok
test_validate (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_validate) ... ok

FAIL: test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 559, in test_implementation
    self.assertCalling(ans.classification.train_epoch, ['train'])
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/__init__.py", line 82, in assertCalling
    self.assertIn(name, call_names, msg=f"Function {func.__qualname__} should call {name}")
AssertionError: 'train' not found in ['train_step', 'item', 'sum', 'argmax'] : Fun

<unittest.runner.TextTestResult run=3 errors=0 failures=1>

## (Bonus) Využití GPU

- Třída `Module` obsahuje metodu `to(dtype, device)`, kde je možné celý model převést na GPU (např. `device='cuda')`).
- Data ale loader `ans.data.BatchLoader` dodává na CPU.
- Pokud se pokusíte např. pronásobit dva tensory, kde každý je na jiném zařízení, PyTorch vyhodí chybu.
- Vstupy se proto musí rovněž přenést na GPU.
- Po vzoru [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) nebudeme chtít řešit převod v `train_step` a `val_step`, které by měly zůstat co nejvíce agnostické vůči technikáliím.
- Upravíme namísto toho `BatchLoader.__iter__`.
  - Třída nově obsahuje parametr `device: str`.
  - Pokud `device != 'cpu'`, převeďte tensory ještě před `yield` na zadanou `device` (metoda `.to()`).

### TODO: upravte metodu [`ans.data.BatchLoader.__iter__`](../ans/data.py).

In [17]:
test_neural_library.TestBatchLoaderWithDevice.eval()

test_batch_size_even (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_batch_size_even) ... ok
test_batch_size_uneven (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_batch_size_uneven) ... ok
test_defaults (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_defaults) ... ok
test_device (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_device) ... FAIL
test_output_tuple_supervised (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_output_tuple_supervised) ... ok
test_output_tuple_unsupervised (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_output_tuple_unsupervised) ... ok
test_shuffle (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_shuffle) ... ok

FAIL: test_device (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_device)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 584, in te

<unittest.runner.TextTestResult run=7 errors=0 failures=1>

## Preprocessing

- Oproti minulým cvičením můžete použít libovolný preprocessing.
- Zároveň urychlíme trénování tak, že upravíme celý dataset pouze jednou a nebude se tak zbytečně opakovat preprocessing v každé dávce.
- Níže je třída `DataPreprocessor`, která má po vzoru [scikit-learn](https://scikit-learn.org/stable/) dvě metody
  1. `fit`
     - převezme `CIFAR10` dataset a spočítá si z něj potřebné statistiky (např. průměr, rozptyl, ...)
     - metoda nemusí dělat nic (např. pokud je celý preprocessing pouze vydělení 255)
  2. `transform`
     - vezme `dataset.data: np.ndarray` a vytvoří z něj `torch.utils.data.TensorDataset`
     - parametr `train: bool` říká, zda se jedná o trénovací sadu či validační (u trénovací můžete využít augmentaci)
- V trénovací části je pak použita jako
  ``` python
  preprocessor = DataPreprocessor()
  preprocessor.fit(train_dataset)
  train_dataset = preprocessor.transform(train_dataset, train=True)
  val_dataset = preprocessor.transform(val_dataset, train=False)
  ```
- Zkuste např. data vycentrovat (stačí např. jednoduché odečtení poloviny rozsahu) a pozorujte, zda se zlepší skóre či ne.

In [18]:
class DataPreprocessor:

    def fit(self, dataset: torchvision.datasets.CIFAR10) -> None:
        ########################################
        # TODO: implement if needed
    
        pass
    
        # ENDTODO
        ########################################

    def transform(self, dataset: torchvision.datasets.CIFAR10, train: bool = False) -> torch.utils.data.TensorDataset:
        ########################################
        # TODO: implement

        x = torch.from_numpy(dataset.data).float()
        x = x.reshape(x.size(0), -1)
        x /= 255
        y = torch.tensor(dataset.targets)

        # ENDTODO
        ########################################

        return torch.utils.data.TensorDataset(x, y)

In [19]:
test_neural_library.TestDataPreprocessor.eval(preprocessor_cls=DataPreprocessor)

test_preprocess (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_preprocess) ... ok

----------------------------------------------------------------------
Ran 1 test in 2.102s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Trénování klasifikátoru

- Využijte vrstvy a optimizéry definované výše k dosažení co nejlepšího skóre.
- Bonusové funkce, vrstvy a optimizéry usnadní dosažení skóre přes 55 % (příp. 60 %) a zároveň urychlí trénování ve smyslu potřebných epoch.
- Nejlepší model uložte jako
  ``` python
  model.save('../output/autograd_classifier.pt')
  ```

### TODO: Natrénujte klasifikační model, který dosáhne alespoň 45 % (bonusově **55 %** a **60 %**) *validační* accuracy.

In [20]:
ans.utils.seed_everything(0)

# hyperparametry
num_epochs = ...
batch_size = ...
learning_rate = ...
weight_decay = ...
device = 'cpu'

# dataset
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True)
val_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True)

# preprocessing
preprocessor = DataPreprocessor()
preprocessor.fit(train_dataset)
train_dataset = preprocessor.transform(train_dataset, train=True)
val_dataset = preprocessor.transform(val_dataset, train=False)

# loadery
train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True, device=device)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False, device=device)

# model
# backbone = ...
# backbone = backbone.to(device=device)
# optimizer = ...
# model = ans.classification.AutogradClassifier(backbone, optimizer)

# ...

Files already downloaded and verified
Files already downloaded and verified


In [21]:
test_neural_library.TestValAccuracy45.eval(preprocessor_cls=DataPreprocessor)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ERROR

ERROR: test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 621, in test_val_acc
    model = ans.classification.AutogradClassifier.load('../output/autograd_classifier.pt')
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../ans/classification.py", line 445, in load
    return torch.load(filename, weights_only=False)  # not the correct & safe way to do this
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/.venv/lib/python3.12/site-packages/torch/serializat

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

In [22]:
test_neural_library.TestValAccuracy55.eval(preprocessor_cls=DataPreprocessor)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ERROR

ERROR: test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 621, in test_val_acc
    model = ans.classification.AutogradClassifier.load('../output/autograd_classifier.pt')
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../ans/classification.py", line 445, in load
    return torch.load(filename, weights_only=False)  # not the correct & safe way to do this
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/.venv/lib/python3.12/site-packages/torch/serializat

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

In [23]:
test_neural_library.TestValAccuracy60.eval(preprocessor_cls=DataPreprocessor)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ERROR

ERROR: test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../tests/test_neural_library.py", line 621, in test_val_acc
    model = ans.classification.AutogradClassifier.load('../output/autograd_classifier.pt')
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/assignments/../ans/classification.py", line 445, in load
    return torch.load(filename, weights_only=False)  # not the correct & safe way to do this
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kirillefremov/development/PycharmProjects/TUL-ANS-2024-25/.venv/lib/python3.12/site-packages/torch/serializat

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

#